In [0]:
%run ./_common

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType,
)

log = get_logger("opspulse.silver")

# Event timestamps are UTC; keep the session in UTC so hour/date extraction is consistent
try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")
except Exception as e:
    log.warning("could not set session time zone: %s", e)

# Only the fields we flatten. Missing fields become null, unknown fields are ignored.
EVENT_SCHEMA = StructType([
    StructField("id", StringType()),
    StructField("type", StringType()),
    StructField("public", BooleanType()),
    StructField("created_at", StringType()),
    StructField("actor", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
    ])),
    StructField("repo", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("org", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
    ])),
    StructField("payload", StructType([
        StructField("action", StringType()),
        StructField("ref", StringType()),
        StructField("ref_type", StringType()),
        StructField("size", LongType()),
        StructField("distinct_size", LongType()),
    ])),
])

In [0]:
def parse_events(bronze_df):
    """Parse the raw JSON string into a struct and keep the raw payload as JSON text."""
    return (
        bronze_df
        .withColumn("e", F.from_json("raw_json", EVENT_SCHEMA))
        .withColumn("payload_json", F.get_json_object("raw_json", "$.payload"))
    )


def flatten_events(parsed_df):
    """Flatten nested actor / repo / org / payload fields into top-level columns."""
    return parsed_df.select(
        F.col("e.id").alias("event_id"),
        F.col("e.type").alias("event_type"),
        F.col("e.public").alias("is_public"),
        F.try_to_timestamp(
            F.col("e.created_at"), F.lit("yyyy-MM-dd'T'HH:mm:ss'Z'")
        ).alias("created_at"),
        F.col("e.actor.id").alias("actor_id"),
        F.col("e.actor.login").alias("actor_login"),
        F.col("e.repo.id").alias("repo_id"),
        F.col("e.repo.name").alias("repo_name"),
        F.col("e.org.id").alias("org_id"),
        F.col("e.org.login").alias("org_login"),
        F.col("e.payload.action").alias("payload_action"),
        F.col("e.payload.ref_type").alias("payload_ref_type"),
        F.col("e.payload.size").alias("push_size"),
        F.col("e.payload.distinct_size").alias("push_distinct_size"),
        "payload_json",
        "source_file",
        F.col("ingest_ts").alias("bronze_ingest_ts"),
    )


def add_bot_flag(df):
    """Flag bot accounts: '[bot]' suffix or a trailing '-bot' / '_bot' / 'bot' token."""
    login = F.lower(F.coalesce(F.col("actor_login"), F.lit("")))
    return df.withColumn(
        "is_bot", login.endswith("[bot]") | login.rlike(r"(^|[-_])bot$")
    )


# A row is valid only if all key business columns are present
VALID_CONDITION = (
    F.col("event_id").isNotNull()
    & F.col("event_type").isNotNull()
    & F.col("created_at").isNotNull()
    & F.col("repo_id").isNotNull()
    & F.col("actor_login").isNotNull()
)


def build_silver(bronze_df):
    """Bronze rows -> clean Silver rows. Returns (silver_df, stats dict)."""
    flagged = add_bot_flag(flatten_events(parse_events(bronze_df)))

    stats = flagged.agg(
        F.count("*").alias("total"),
        F.sum((~VALID_CONDITION).cast("int")).alias("invalid"),
        F.sum((VALID_CONDITION & F.col("is_bot")).cast("int")).alias("bots"),
    ).first().asDict()

    silver = (
        flagged
        .filter(VALID_CONDITION)
        .filter(~F.col("is_bot"))
        .dropDuplicates(["event_id"])
        .withColumn("event_date", F.to_date("created_at"))
        .withColumn("event_hour", F.hour("created_at"))
    )
    return silver, stats

In [0]:
def write_silver(silver_df) -> None:
    """Create the table on first run, otherwise insert-only MERGE on event_id."""
    if not spark.catalog.tableExists(SILVER_TABLE):
        (
            silver_df.write
            .format("delta")
            .partitionBy("event_date")
            .saveAsTable(SILVER_TABLE)
        )
        log.info("created %s", SILVER_TABLE)
        return
    (
        DeltaTable.forName(spark, SILVER_TABLE).alias("t")
        .merge(
            silver_df.alias("s"),
            "t.event_id = s.event_id AND t.event_date = s.event_date",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


def run_silver() -> int:
    """Incremental Bronze -> Silver. Returns the number of new Silver rows."""
    bronze = spark.table(BRONZE_TABLE)

    if spark.catalog.tableExists(SILVER_TABLE):
        last_ts = spark.table(SILVER_TABLE).agg(F.max("bronze_ingest_ts")).first()[0]
        if last_ts is not None:
            bronze = bronze.filter(F.col("ingest_ts") > F.lit(last_ts))

    if bronze.isEmpty():
        log.info("no new bronze rows, nothing to do")
        return 0

    before = spark.table(SILVER_TABLE).count() if spark.catalog.tableExists(SILVER_TABLE) else 0
    try:
        silver_df, stats = build_silver(bronze)
        log.info(
            "bronze rows: %d | invalid: %d | bots dropped: %d",
            stats["total"], stats["invalid"], stats["bots"],
        )
        write_silver(silver_df)
    except Exception as e:
        log.error("silver run failed: %s", e)
        raise

    after = spark.table(SILVER_TABLE).count()
    log.info("silver finished: %d new rows, %d total", after - before, after)
    return after - before


run_silver()

In [0]:
silver = spark.table(SILVER_TABLE)
print("silver rows:", silver.count())

# 1) Duplicates
dupes = silver.groupBy("event_id").count().filter("count > 1").count()
print("duplicate event_ids:", dupes)

# 2) Nulls in key columns
key_cols = ["event_id", "event_type", "created_at", "repo_id", "actor_login", "event_date", "event_hour"]
display(silver.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in key_cols]))

# 3) Bots left behind (must be 0)
print("bot rows in silver:", silver.filter(F.col("is_bot") == True).count() if "is_bot" in silver.columns else "column not stored (bots dropped before write)")

# 4) Distributions
display(silver.groupBy("event_type").count().orderBy(F.desc("count")))
display(silver.groupBy("event_date", "event_hour").count().orderBy("event_date", "event_hour"))
display(silver.agg(F.min("created_at").alias("min_ts"), F.max("created_at").alias("max_ts")))

# 5) Flattened payload fields: how populated are they?
display(silver.agg(
    F.avg(F.col("push_size").isNotNull().cast("int")).alias("push_size_fill_rate"),
    F.avg(F.col("payload_action").isNotNull().cast("int")).alias("action_fill_rate"),
))